# Validación de modelos V5 para la app Streamlit

Este notebook valida que la carpeta `modelos` que usará `app.py` sea compatible con la versión de selección automática de modelos por producto.

Revisa:

- `config_entrenamiento.pkl`
- `config_entrenamiento.json`
- `modelo_almuerzo.pkl`
- `modelo_sopa.pkl`
- `modelo_fanesca.pkl`
- `modelo_colada_morada.pkl`

Además genera una predicción de prueba para confirmar que todos los modelos pueden ejecutarse correctamente.

## Bloque 1. Importar librerías y definir rutas

In [4]:
# ============================================================
# BLOQUE 1. IMPORTAR LIBRERÍAS Y DEFINIR RUTAS
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
import json
import joblib
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# 1. RUTA PRINCIPAL DE GOOGLE DRIVE
# ============================================================

MI_DRIVE = Path("/content/drive/MyDrive")

print("Existe MyDrive:", MI_DRIVE.exists())
print("Ruta MyDrive:", MI_DRIVE)

if not MI_DRIVE.exists():
    raise FileNotFoundError(
        "No se encontró /content/drive/MyDrive. "
        "Verifica que Google Drive esté montado correctamente."
    )

# ============================================================
# 2. BUSCAR CARPETA DE MODELOS
# ============================================================

# Ruta esperada según tu captura:
RUTA_MODELOS = MI_DRIVE / "modelos_mejor_modelo"

# Si no existe, se busca automáticamente en MyDrive
if not RUTA_MODELOS.exists():
    print("No se encontró la carpeta en la ruta esperada.")
    print("Buscando carpeta 'modelos_mejor_modelo' dentro de MyDrive...")

    carpetas_encontradas = list(MI_DRIVE.rglob("modelos_mejor_modelo"))

    if carpetas_encontradas:
        RUTA_MODELOS = carpetas_encontradas[0]
        print("Carpeta encontrada automáticamente:")
        print(RUTA_MODELOS)
    else:
        print("Carpetas disponibles en MyDrive:")
        for item in MI_DRIVE.iterdir():
            if item.is_dir():
                print("-", item.name)

        raise FileNotFoundError(
            "No se encontró la carpeta 'modelos_mejor_modelo'. "
            "Verifica el nombre exacto de la carpeta en Google Drive."
        )

# ============================================================
# 3. DEFINIR ARCHIVOS DE CONFIGURACIÓN
# ============================================================

RUTA_CONFIG_PKL = RUTA_MODELOS / "config_entrenamiento.pkl"
RUTA_CONFIG_JSON = RUTA_MODELOS / "config_entrenamiento.json"

PRODUCTOS_ESPERADOS = ["almuerzo", "sopa", "fanesca", "colada_morada"]

# ============================================================
# 4. VALIDAR ARCHIVOS ESPERADOS
# ============================================================

archivos_esperados = [
    "config_entrenamiento.pkl",
    "config_entrenamiento.json",
    "modelo_almuerzo.pkl",
    "modelo_sopa.pkl",
    "modelo_fanesca.pkl",
    "modelo_colada_morada.pkl"
]

print("\nCarpeta de modelos:", RUTA_MODELOS)
print("Existe carpeta:", RUTA_MODELOS.exists())

print("\nArchivos encontrados en la carpeta:")
for archivo in RUTA_MODELOS.iterdir():
    print("-", archivo.name)

faltantes = []

for archivo in archivos_esperados:
    ruta_archivo = RUTA_MODELOS / archivo
    if not ruta_archivo.exists():
        faltantes.append(archivo)

if faltantes:
    raise FileNotFoundError(
        "Faltan estos archivos dentro de la carpeta modelos_mejor_modelo:\n"
        + "\n".join(f"- {archivo}" for archivo in faltantes)
    )

print("\nValidación correcta: todos los archivos necesarios existen.")

Mounted at /content/drive
Existe MyDrive: True
Ruta MyDrive: /content/drive/MyDrive

Carpeta de modelos: /content/drive/MyDrive/modelos_mejor_modelo
Existe carpeta: True

Archivos encontrados en la carpeta:
- modelo_almuerzo.pkl
- modelo_sopa.pkl
- modelo_fanesca.pkl
- modelo_colada_morada.pkl
- config_entrenamiento.pkl
- config_entrenamiento.json

Validación correcta: todos los archivos necesarios existen.


## Bloque 2. Cargar configuración PKL + JSON

In [5]:
CONFIG_DEFAULT = {
    "productos": PRODUCTOS_ESPERADOS,
    "variables_predictoras": [],
    "fecha_min_modelo": "2023-01-02",
    "fecha_max_modelo": "2025-12-31",
    "temporada_fanesca": {"meses": [2, 3]},
    "temporada_colada_morada": {"inicio_mes": 10, "inicio_dia": 1, "fin_mes": 11, "fin_dia": 4},
    "base_ciclo_dia_semana": 5,
    "modelo_por_producto": {}
}

config = CONFIG_DEFAULT.copy()

if RUTA_CONFIG_PKL.exists():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        config_pkl = joblib.load(RUTA_CONFIG_PKL)
    if isinstance(config_pkl, dict):
        config.update(config_pkl)

if RUTA_CONFIG_JSON.exists():
    with open(RUTA_CONFIG_JSON, "r", encoding="utf-8") as f:
        config_json = json.load(f)
    if isinstance(config_json, dict):
        config.update(config_json)

PRODUCTOS = config.get("productos", PRODUCTOS_ESPERADOS)
VARIABLES = config.get("variables_predictoras", [])
MODELO_POR_PRODUCTO = config.get("modelo_por_producto", {})
BASE_CICLO_DIA_SEMANA = int(config.get("base_ciclo_dia_semana", 5))

print("Versión:", config.get("version"))
print("Modelo general:", config.get("modelo"))
print("Productos:", PRODUCTOS)
print("Variables predictoras:", len(VARIABLES))
print("Base ciclo día semana:", BASE_CICLO_DIA_SEMANA)
print("Modelo por producto:")
print(MODELO_POR_PRODUCTO)

Versión: Version_5_seleccion_automatica
Modelo general: Seleccion_automatica_por_producto
Productos: ['almuerzo', 'sopa', 'fanesca', 'colada_morada']
Variables predictoras: 26
Base ciclo día semana: 5
Modelo por producto:
{'almuerzo': 'HistGradientBoosting', 'sopa': 'ElasticNet', 'fanesca': 'HistGradientBoosting', 'colada_morada': 'RandomForest'}


## Bloque 3. Cargar modelos y validar consistencia

In [6]:
def obtener_variables_modelo(modelo):
    if hasattr(modelo, "feature_names_in_"):
        return list(modelo.feature_names_in_)
    return list(VARIABLES)


def obtener_tipo_modelo_real(modelo):
    if hasattr(modelo, "steps") and getattr(modelo, "steps"):
        return "Pipeline → " + type(modelo.steps[-1][1]).__name__
    return type(modelo).__name__


modelos = {}
diagnostico = []

for producto in PRODUCTOS:
    ruta_modelo = RUTA_MODELOS / f"modelo_{producto}.pkl"

    if not ruta_modelo.exists():
        diagnostico.append({
            "producto": producto,
            "archivo": str(ruta_modelo),
            "estado": "No encontrado",
            "modelo_config": MODELO_POR_PRODUCTO.get(producto, "No definido"),
            "modelo_real": None,
            "variables_modelo": None,
            "variables_config": len(VARIABLES)
        })
        continue

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        modelo = joblib.load(ruta_modelo)

    modelos[producto] = modelo

    modelo_real = obtener_tipo_modelo_real(modelo)
    modelo_config = MODELO_POR_PRODUCTO.get(producto, "No definido")
    variables_modelo = obtener_variables_modelo(modelo)

    diagnostico.append({
        "producto": producto,
        "archivo": str(ruta_modelo),
        "estado": "Cargado",
        "modelo_config": modelo_config,
        "modelo_real": modelo_real,
        "coincide_config_pkl": "Sí" if modelo_config in modelo_real or modelo_real in modelo_config else "Revisar",
        "variables_modelo": len(variables_modelo),
        "variables_config": len(VARIABLES)
    })

df_diagnostico = pd.DataFrame(diagnostico)
display(df_diagnostico)

if (df_diagnostico["estado"] != "Cargado").any():
    raise FileNotFoundError("Faltan modelos. Revisa la columna estado.")

if (df_diagnostico["variables_modelo"] != df_diagnostico["variables_config"]).any():
    print("Advertencia: hay diferencias entre variables del modelo y variables del config.")

if "coincide_config_pkl" in df_diagnostico.columns and (df_diagnostico["coincide_config_pkl"] == "Revisar").any():
    print("Advertencia: algunos PKL no parecen coincidir con el modelo indicado en config_entrenamiento.json.")

,producto,archivo,estado,modelo_config,modelo_real,coincide_config_pkl,variables_modelo,variables_config
0,almuerzo,/content/drive/MyDrive/modelos_mejor_modelo/mo...,Cargado,HistGradientBoosting,HistGradientBoostingRegressor,Sí,26,26
1,sopa,/content/drive/MyDrive/modelos_mejor_modelo/mo...,Cargado,ElasticNet,Pipeline → ElasticNet,Sí,26,26
2,fanesca,/content/drive/MyDrive/modelos_mejor_modelo/mo...,Cargado,HistGradientBoosting,HistGradientBoostingRegressor,Sí,26,26
3,colada_morada,/content/drive/MyDrive/modelos_mejor_modelo/mo...,Cargado,RandomForest,RandomForestRegressor,Sí,26,26


## Bloque 4. Crear datos futuros de prueba

In [7]:
FECHA_MIN_MODELO = pd.to_datetime(config.get("fecha_min_modelo", "2023-01-02"))

def es_fanesca_temporada(fecha):
    meses = config.get("temporada_fanesca", {}).get("meses", [2, 3])
    return int(fecha.month in meses)


def es_colada_temporada(fecha):
    temp = config.get("temporada_colada_morada", {
        "inicio_mes": 10, "inicio_dia": 1, "fin_mes": 11, "fin_dia": 4
    })

    inicio = pd.Timestamp(year=fecha.year, month=int(temp["inicio_mes"]), day=int(temp["inicio_dia"]))
    fin = pd.Timestamp(year=fecha.year, month=int(temp["fin_mes"]), day=int(temp["fin_dia"]))

    return int(inicio <= fecha <= fin)


fechas = pd.date_range("2026-01-05", periods=30, freq="B")

df_prueba = pd.DataFrame({"fecha": fechas})
df_prueba["anio"] = df_prueba["fecha"].dt.year
df_prueba["mes"] = df_prueba["fecha"].dt.month
df_prueba["dia_mes"] = df_prueba["fecha"].dt.day
df_prueba["dia_semana_num"] = df_prueba["fecha"].dt.weekday
df_prueba["semana_anio"] = df_prueba["fecha"].dt.isocalendar().week.astype(int)

df_prueba["es_fanesca_temporada"] = df_prueba["fecha"].apply(es_fanesca_temporada)
df_prueba["es_colada_temporada"] = df_prueba["fecha"].apply(es_colada_temporada)

df_prueba["es_inicio_mes"] = (df_prueba["dia_mes"] <= 5).astype(int)
df_prueba["es_quincena"] = df_prueba["dia_mes"].between(13, 17).astype(int)
df_prueba["es_fin_mes"] = (df_prueba["dia_mes"] >= 25).astype(int)

df_prueba["es_lunes"] = (df_prueba["dia_semana_num"] == 0).astype(int)
df_prueba["es_martes"] = (df_prueba["dia_semana_num"] == 1).astype(int)
df_prueba["es_miercoles"] = (df_prueba["dia_semana_num"] == 2).astype(int)
df_prueba["es_jueves"] = (df_prueba["dia_semana_num"] == 3).astype(int)
df_prueba["es_viernes"] = (df_prueba["dia_semana_num"] == 4).astype(int)

df_prueba["mes_sin"] = np.sin(2 * np.pi * df_prueba["mes"] / 12)
df_prueba["mes_cos"] = np.cos(2 * np.pi * df_prueba["mes"] / 12)
df_prueba["dia_semana_sin"] = np.sin(2 * np.pi * df_prueba["dia_semana_num"] / BASE_CICLO_DIA_SEMANA)
df_prueba["dia_semana_cos"] = np.cos(2 * np.pi * df_prueba["dia_semana_num"] / BASE_CICLO_DIA_SEMANA)

df_prueba["tendencia"] = (df_prueba["fecha"] - FECHA_MIN_MODELO).dt.days.clip(lower=0)
df_prueba["tendencia_log"] = np.log1p(df_prueba["tendencia"])
df_prueba["crecimiento_anual"] = df_prueba["anio"] - FECHA_MIN_MODELO.year

df_prueba["preciomenu"] = 5.0
df_prueba["preciosopa"] = 2.0
df_prueba["fanesca_precio"] = 10.0
df_prueba["coladamorada_precio"] = 3.5

for var in VARIABLES:
    if var not in df_prueba.columns:
        df_prueba[var] = 0

display(df_prueba[["fecha"] + VARIABLES].head())

,fecha,anio,mes,dia_mes,dia_semana_num,semana_anio,es_fanesca_temporada,es_colada_temporada,es_inicio_mes,es_quincena,...,mes_cos,dia_semana_sin,dia_semana_cos,tendencia,tendencia_log,crecimiento_anual,preciomenu,preciosopa,fanesca_precio,coladamorada_precio
0,2026-01-05,2026,1,5,0,2,0,0,1,0,...,0.866025,0.000000,1.000000,1099,7.003065,3,5.0,2.0,10.0,3.5
1,2026-01-06,2026,1,6,1,2,0,0,0,0,...,0.866025,0.951057,0.309017,1100,7.003974,3,5.0,2.0,10.0,3.5
2,2026-01-07,2026,1,7,2,2,0,0,0,0,...,0.866025,0.587785,-0.809017,1101,7.004882,3,5.0,2.0,10.0,3.5
3,2026-01-08,2026,1,8,3,2,0,0,0,0,...,0.866025,-0.587785,-0.809017,1102,7.005789,3,5.0,2.0,10.0,3.5
4,2026-01-09,2026,1,9,4,2,0,0,0,0,...,0.866025,-0.951057,0.309017,1103,7.006695,3,5.0,2.0,10.0,3.5


## Bloque 5. Ejecutar predicción de prueba

In [8]:
predicciones = df_prueba[["fecha"]].copy()

for producto, modelo in modelos.items():
    variables_modelo = obtener_variables_modelo(modelo)

    faltantes = [v for v in variables_modelo if v not in df_prueba.columns]
    if faltantes:
        raise ValueError(f"Faltan variables para {producto}: {faltantes}")

    pred = modelo.predict(df_prueba[variables_modelo].fillna(0))
    pred = np.maximum(pred, 0)

    if producto == "fanesca":
        pred = np.where(df_prueba["es_fanesca_temporada"].to_numpy() == 1, pred, 0)

    if producto == "colada_morada":
        pred = np.where(df_prueba["es_colada_temporada"].to_numpy() == 1, pred, 0)

    predicciones[producto] = np.round(pred).astype(int)

display(predicciones.head(15))
print("Predicción de prueba ejecutada correctamente.")

,fecha,almuerzo,sopa,fanesca,colada_morada
0,2026-01-05,98,79,0,0
1,2026-01-06,121,58,0,0
2,2026-01-07,118,57,0,0
3,2026-01-08,114,57,0,0
4,2026-01-09,103,79,0,0
5,2026-01-12,101,78,0,0
6,2026-01-13,126,55,0,0
7,2026-01-14,118,54,0,0
8,2026-01-15,116,55,0,0
9,2026-01-16,97,76,0,0


Predicción de prueba ejecutada correctamente.


## Bloque 6. Exportar reporte y ZIP para la app

In [9]:
OUTPUT_REPORTE = RUTA_MODELOS / "reporte_validacion_app_multimodelo.xlsx"
OUTPUT_ZIP = RUTA_MODELOS / "modelos_para_app.zip"

with pd.ExcelWriter(OUTPUT_REPORTE, engine="openpyxl") as writer:
    df_diagnostico.to_excel(writer, sheet_name="diagnostico_modelos", index=False)
    predicciones.to_excel(writer, sheet_name="prediccion_prueba", index=False)

    df_config = pd.DataFrame([
        {"campo": k, "valor": json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v}
        for k, v in config.items()
    ])
    df_config.to_excel(writer, sheet_name="config", index=False)

archivos_zip = [
    "config_entrenamiento.pkl",
    "config_entrenamiento.json",
    "modelo_almuerzo.pkl",
    "modelo_sopa.pkl",
    "modelo_fanesca.pkl",
    "modelo_colada_morada.pkl"
]

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for archivo in archivos_zip:
        ruta = RUTA_MODELOS / archivo
        if ruta.exists():
            z.write(ruta, arcname=f"modelos/{archivo}")

print("Reporte generado:", OUTPUT_REPORTE)
print("ZIP generado:", OUTPUT_ZIP)
print("Estructura esperada para la app:")
print("app.py")
print("modelos/")
for archivo in archivos_zip:
    print("  -", archivo)

Reporte generado: /content/drive/MyDrive/modelos_mejor_modelo/reporte_validacion_app_multimodelo.xlsx
ZIP generado: /content/drive/MyDrive/modelos_mejor_modelo/modelos_para_app.zip
Estructura esperada para la app:
app.py
modelos/
  - config_entrenamiento.pkl
  - config_entrenamiento.json
  - modelo_almuerzo.pkl
  - modelo_sopa.pkl
  - modelo_fanesca.pkl
  - modelo_colada_morada.pkl
